# Ungraded Lab: Quantization and Pruning with Random Forest on Titanic

In this lab you will apply model compression techniques to a **RandomForestClassifier** trained on the **Titanic** dataset:
- **Post-training quantization** (feature discretization)
- **Quantization-aware training** (train with discretized features)
- **Weight pruning** (reduce tree complexity via depth/sample constraints)

Let's begin!

## Imports

Let's first import libraries and define utility functions.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import os, pickle, tempfile, zipfile
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder, KBinsDiscretizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

## Utilities and Constants

In [ ]:
# String constants for model filenames
FILE_BASELINE_PKL     = 'baseline_model.pkl'
FILE_QUANTIZED_PKL    = 'quantized_model.pkl'
FILE_QAT_PKL          = 'qat_model.pkl'
FILE_PRUNED_PKL       = 'pruned_model.pkl'
FILE_PRUNED_QUANT_PKL = 'pruned_quantized_model.pkl'

MODEL_SIZE = {}
ACCURACY   = {}

def print_metric(metric_dict, metric_name):
    for metric, value in metric_dict.items():
        if isinstance(value, float):
            print(f'{metric_name} for {metric}: {value:.4f}')
        else:
            print(f'{metric_name} for {metric}: {value}')

def save_model(model, filename):
    with open(filename, 'wb') as f:
        pickle.dump(model, f)

def load_model(filename):
    with open(filename, 'rb') as f:
        return pickle.load(f)

def get_gzipped_model_size(file):
    _, zipped = tempfile.mkstemp('.zip')
    with zipfile.ZipFile(zipped, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(file)
    return os.path.getsize(zipped)

def evaluate_model(model, X_test, y_test):
    return accuracy_score(y_test, model.predict(X_test))

def model_builder():
    return RandomForestClassifier(n_estimators=100, random_state=42, criterion='entropy')

## Download and Prepare the Dataset

You will use the Titanic dataset. The helper functions are made to work with this dataset.

In [ ]:
def load_titanic():
    df = sns.load_dataset('titanic')
    df.drop(['class','who','adult_male','deck','embark_town','alive','alone'], axis=1, inplace=True)
    df['age'].fillna(df['age'].median(), inplace=True)
    df['embarked'].fillna(df['embarked'].mode()[0], inplace=True)
    df['sex'] = df['sex'].map({'male':1,'female':0})
    df['embarked'] = LabelEncoder().fit_transform(df['embarked'])
    X = df.drop('survived', axis=1)
    y = df['survived']
    return train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

X_train, X_test, y_train, y_test = load_titanic()
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)
print(f"Train: {X_train_s.shape}, Test: {X_test_s.shape}")

## Baseline Model

Build and train a standard RandomForestClassifier. This is the baseline for comparison.

In [ ]:
baseline_model = model_builder()
baseline_model.fit(X_train_s, y_train)
baseline_model.save_weights = lambda f: save_model(baseline_model, f)  # compatibility alias
save_model(baseline_model, FILE_BASELINE_PKL)
print("Baseline model trained and saved.")
print(f"n_estimators: {baseline_model.n_estimators}, max_depth: {baseline_model.max_depth}")

In [ ]:
_, ACCURACY['baseline model'] = None, evaluate_model(baseline_model, X_test_s, y_test)
MODEL_SIZE['baseline pkl'] = os.path.getsize(FILE_BASELINE_PKL)
print_metric(ACCURACY, 'test accuracy')
print_metric(MODEL_SIZE, 'model size in bytes')

### Serialize the Model

Save the model as a pickle file and record its size.

In [ ]:
save_model(baseline_model, FILE_BASELINE_PKL)
MODEL_SIZE['baseline pkl'] = os.path.getsize(FILE_BASELINE_PKL)
print_metric(MODEL_SIZE, 'model size in bytes')

## Post-Training Quantization

Analogous to float32→int8 quantization in neural networks, we discretize continuous features into integer bins. This reduces the precision of feature representations.

In [ ]:
# Discretize features into n_bins integer bins (post-training quantization)
def quantize_features(X_train, X_test, n_bins=8):
    discretizer = KBinsDiscretizer(n_bins=n_bins, encode='ordinal', strategy='quantile')
    X_train_q = discretizer.fit_transform(X_train)
    X_test_q  = discretizer.transform(X_test)
    return X_train_q, X_test_q, discretizer

X_train_q, X_test_q, discretizer = quantize_features(X_train_s, X_test_s, n_bins=8)
print(f"Feature range before: min={X_train_s.min():.2f}, max={X_train_s.max():.2f}")
print(f"Feature range after:  min={X_train_q.min():.0f}, max={X_train_q.max():.0f}")

In [ ]:
# Apply quantized features to baseline model
ACCURACY['post-training quantized'] = evaluate_model(baseline_model, X_test_q, y_test)
print_metric(ACCURACY, 'test accuracy')

You should see around the same or slightly different accuracy. This comes from reducing the floating-point precision into integer bins.

## Quantization Aware Training

Train the model from scratch using the quantized (discretized) features, so the model adapts to the reduced precision — analogous to Quantization Aware Training (QAT) in neural networks.

In [ ]:
qat_model = model_builder()
qat_model.fit(X_train_q, y_train)
save_model(qat_model, FILE_QAT_PKL)

ACCURACY['quantization aware non-quantized'] = evaluate_model(qat_model, X_test_s, y_test)
ACCURACY['quantization aware quantized']     = evaluate_model(qat_model, X_test_q, y_test)
print_metric(ACCURACY, 'test accuracy')

The QAT model trains with quantized features so it learns to adapt to precision loss — typically yielding better accuracy on quantized inputs than post-training quantization.

## Pruning

Pruning reduces model complexity by constraining tree growth: `max_depth` limits tree depth, `min_samples_split` and `min_samples_leaf` prevent overfitting. This is analogous to weight pruning (zeroing low-magnitude weights) in neural networks.

In [ ]:
# Pruning parameters — equivalent to PolynomialDecay sparsity schedule
pruning_params = {
    'max_depth':          5,    # limit tree depth
    'min_samples_split':  10,   # require 10+ samples to split
    'min_samples_leaf':   5,    # require 5+ samples in leaf
    'max_features':       'sqrt',# use sqrt(n_features) per split
    'n_estimators':       50    # fewer trees
}

pruned_model = RandomForestClassifier(**pruning_params, random_state=42, criterion='entropy')
pruned_model.fit(X_train_s, y_train)

save_model(pruned_model, FILE_PRUNED_PKL)
print(f"Pruned model: {pruning_params}")

In [ ]:
# Preview number of nodes in first tree before vs after pruning
print(f"Baseline tree node count (tree 0): {baseline_model.estimators_[0].tree_.node_count}")
print(f"Pruned   tree node count (tree 0): {pruned_model.estimators_[0].tree_.node_count}")

In [ ]:
MODEL_SIZE = {}
MODEL_SIZE['baseline pkl']     = os.path.getsize(FILE_BASELINE_PKL)
MODEL_SIZE['pruned non-quantized pkl'] = os.path.getsize(FILE_PRUNED_PKL)
print_metric(MODEL_SIZE, 'model size in bytes')

In [ ]:
# Compressed size comparison
MODEL_SIZE = {}
MODEL_SIZE['baseline pkl']     = get_gzipped_model_size(FILE_BASELINE_PKL)
MODEL_SIZE['pruned non-quantized pkl'] = get_gzipped_model_size(FILE_PRUNED_PKL)
print_metric(MODEL_SIZE, 'gzipped model size in bytes')

The pruned model compresses more efficiently due to its simpler (sparser) decision boundaries.

## Combined Pruning + Quantization

Apply both techniques for maximum compression — analogous to the ~10x combined reduction in the neural network version.

In [ ]:
pruned_qat_model = RandomForestClassifier(**pruning_params, random_state=42, criterion='entropy')
pruned_qat_model.fit(X_train_q, y_train)
save_model(pruned_qat_model, FILE_PRUNED_QUANT_PKL)

MODEL_SIZE['pruned + quantized pkl'] = get_gzipped_model_size(FILE_PRUNED_QUANT_PKL)
print_metric(MODEL_SIZE, 'gzipped model size in bytes')

In [ ]:
ACCURACY = {}
ACCURACY['baseline']             = evaluate_model(baseline_model, X_test_s, y_test)
ACCURACY['post-training quant']  = evaluate_model(baseline_model, X_test_q, y_test)
ACCURACY['qat']                  = evaluate_model(qat_model, X_test_q, y_test)
ACCURACY['pruned']               = evaluate_model(pruned_model, X_test_s, y_test)
ACCURACY['pruned + quantized']   = evaluate_model(pruned_qat_model, X_test_q, y_test)
print_metric(ACCURACY, 'accuracy')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))
names = list(ACCURACY.keys())
accs  = list(ACCURACY.values())
axes[0].barh(names, accs, color='steelblue')
axes[0].set_xlabel('Accuracy')
axes[0].set_title('Accuracy Comparison')
axes[0].set_xlim(0,1)

sizes = {k: get_gzipped_model_size(v) for k,v in [
    ('baseline', FILE_BASELINE_PKL),
    ('qat', FILE_QAT_PKL),
    ('pruned', FILE_PRUNED_PKL),
    ('pruned+quant', FILE_PRUNED_QUANT_PKL)
]}
axes[1].bar(list(sizes.keys()), list(sizes.values()), color='coral')
axes[1].set_ylabel('Gzipped size (bytes)')
axes[1].set_title('Model Size Comparison')
plt.tight_layout()
plt.show()

## Wrap Up

In this notebook you practiced model compression techniques adapted for Random Forest:
- **Post-training quantization** → feature discretization (KBinsDiscretizer)
- **Quantization-aware training** → train with discretized features
- **Pruning** → constrain tree depth and leaf samples
- **Combined** → maximum compression with minimal accuracy loss

For more information:
- [sklearn RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)
- [KBinsDiscretizer](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.KBinsDiscretizer.html)

**Congratulations and enjoy the rest of the course!**